# Manual RAG banayenge without LangChain.

In [4]:
#!pip install pymupdf sentence-transformers faiss-cpu requests tqdm

## STEP 1 — Load 10 PDFs
#### Extraction of text

In [11]:
import os
import fitz  # PyMuPDF
from tqdm import tqdm

In [12]:
DATA_PATH = "./data/"

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

In [13]:
documents = []

for file in tqdm(os.listdir(DATA_PATH)):
    if file.endswith(".pdf"):
        full_path = os.path.join(DATA_PATH, file)
        text = extract_text_from_pdf(full_path)
        documents.append({
            "filename": file,
            "text": text
        })

print("Total documents loaded:", len(documents))

100%|████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:04<00:00,  1.59s/it]

Total documents loaded: 3


In [18]:
documents

[{'filename': 'Atharva Resume.pdf',
  'text': "Atharva Sunil Jadhav \n+91 8291632871 · atharvajadhav2023@gmail.com · LinkedIn    GitHub  \nNavi Mumbai, India \n \nEDUCATION \n \nB.E in Computer Engineering \nMGM College of Engineering and Technology, Kamothe \n \nHigher Secondary Education(HSC)  \nP. E. Society's English Medium High School & Jr. College , Thane \n \nSenior Secondary Education(SSC) \nSt. Xavier’s High School, Navi Mumbai \n  \n \nINTERNSHIP / EXTRA CURRICULAR ACTIVITIES  \nTata Data Visualisation: Empowering Business with Effective Insights Job Simulation on Forage - July 2024 \n\uf0b7 \nCompleted a simulation involving creating data visualizations for Tata Consultancy Services. \n\uf0b7 \nPrepared questions for a meeting with client senior leadership. \n\uf0b7 \nCreated visuals for data analysis to help executives with effective decision making. \n \nPROJECTS \n \nHousing Price Prediction  \nDeveloped a real estate price prediction website utilizing machine learning al

## STEP 2 — Chunking

In [32]:
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap
        
    return chunks

# What is overlap?

# Without overlap:
# Chunk1: ABCDEFG
# Chunk2: HIJKLMN

# With overlap:
# Chunk1: ABCDEFG
# Chunk2: EFGHIJK

# Better semantic continuity.

In [33]:
all_chunks = []

for doc in documents:
    chunks = chunk_text(doc["text"])
    for chunk in chunks:
        all_chunks.append({
            "filename": doc["filename"],
            "content": chunk
        })

print("Total chunks created:", len(all_chunks))

Total chunks created: 5831


In [34]:
print("each document is divided into several chuncks,each chunk has 500 char:",len(all_chunks[0]['content']))

each document is divided into several chuncks,each chunk has 500 char: 500


In [35]:
all_chunks

[{'filename': 'Atharva Resume.pdf',
  'content': "Atharva Sunil Jadhav \n+91 8291632871 · atharvajadhav2023@gmail.com · LinkedIn    GitHub  \nNavi Mumbai, India \n \nEDUCATION \n \nB.E in Computer Engineering \nMGM College of Engineering and Technology, Kamothe \n \nHigher Secondary Education(HSC)  \nP. E. Society's English Medium High School & Jr. College , Thane \n \nSenior Secondary Education(SSC) \nSt. Xavier’s High School, Navi Mumbai \n  \n \nINTERNSHIP / EXTRA CURRICULAR ACTIVITIES  \nTata Data Visualisation: Empowering Business with Effective Insights J"},
 {'filename': 'Atharva Resume.pdf',
  'content': 'EXTRA CURRICULAR ACTIVITIES  \nTata Data Visualisation: Empowering Business with Effective Insights Job Simulation on Forage - July 2024 \n\uf0b7 \nCompleted a simulation involving creating data visualizations for Tata Consultancy Services. \n\uf0b7 \nPrepared questions for a meeting with client senior leadership. \n\uf0b7 \nCreated visuals for data analysis to help executives

## STEP 3 — Create Embeddings

In [51]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [40]:
texts = [chunk["content"] for chunk in all_chunks]
embeddings = model.encode(texts, show_progress_bar=True)
embeddings = np.array(embeddings)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/183 [00:00<?, ?it/s]

Embedding shape: (5831, 384)


## STEP 4 — Create FAISS Index

In [41]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("Total vectors in index:", index.ntotal)

Total vectors in index: 5831


## STEP 5 — Retrieval Function

In [42]:
def retrieve(query, top_k=3):
    query_embedding = model.encode([query])
    query_embedding = np.array(query_embedding)
    
    distances, indices = index.search(query_embedding, top_k)
    
    results = []
    for idx in indices[0]:
        results.append(all_chunks[idx])
    
    return results

##  STEP 6 — Connect Ollama

In [43]:
import requests
import json
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "mistral"

def generate_answer(context, query):
    
    prompt = f"""
You are a helpful assistant.

Use the following context to answer the question.

Context:
{context}

Question:
{query}

Answer:
"""
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "prompt": prompt,
            "stream": False
        }
    )
    
    return response.json()["response"]

## STEP 7 — Full RAG Pipeline

In [45]:
def rag_pipeline(query):
    
    retrieved_chunks = retrieve(query, top_k=3)
    context = "\n\n".join([chunk["content"] for chunk in retrieved_chunks])
    answer = generate_answer(context, query)
    return answer

In [50]:
query = "what are the skills mentioned in my resume"

response = rag_pipeline(query)

print(response)

 The skills mentioned in your resume, as per the provided context, are Machine Learning, AWS Fundamentals, Python, Data Structures and Algorithms, Data Cleaning and Preprocessing, Data Visualization, Deep Learning, Problem-Solving, Decision-making, Teamwork, Communication, Time Management, Agile, and flexibility towards working from any of LTIMindtree's development centers across the country. Additionally, you have certified AI and Data Skills by YBI Foundation.
